In [1]:
import os
import json
import math
import re
from math import exp
import random
from collections import Counter
from typing import List, Dict

import numpy as np
from tqdm import tqdm

import torch
torch._dynamo.disable()

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import mean_squared_error

import csv
import pandas as pd
import matplotlib.pyplot as plt

from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

import csv
from datetime import datetime

# torch._dynamo.disable()

c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

CONFIG = {
    'annotations': r"E:\WLASL\wlasl_1000_preproc\train_final.json",
    'data_root': r'E:\WLASL\wlasl_1000_preproc\videos',

    # --- saving ---
    'save_dir': './checkpoints_text2sign_4_1_wlasl1000',
    'save_every': 1,

    # --- training ---
    'batch_size': 24,
    'epochs': 100,
    'lr': 1e-4,
    'weight_decay': 1e-2,
    'patience': 10,
    'grad_clip': 1.0,
    'teacher_forcing_rate': 0.7,
    'lambda_vel': 0.1,
    'disable_early_stop': False,

    # --- device ---
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'seed': 42,

    # --- text ---
    'max_text_len': 32,
    'pad_id': 0,

    # --- landmarks ---
    'max_landmark_len': 20,
    'num_landmarks': 75,
    'landmark_dim': 3,
    'landmark_input_dim': 75 * 3,  # 225

    # --- transformer ---
    'd_model': 512,
    'nhead': 8,
    'num_encoder_layers': 3,
    'num_decoder_layers': 3,
    'dropout': 0.1,
    
    "custom_accuracy": {
        "enabled": True,

        # how predicted sequence is compared to GT instances of same gloss
        # "best"  -> min distance across all .npy files of that gloss
        # "mean"  -> mean distance across all .npy files
        "match_mode": "best",

        # how distance is converted to accuracy
        # "exp"        -> exp(-dist / alpha)
        # "threshold"  -> dist <= threshold ? 1 : 0
        # "linear"     -> max(0, 1 - dist / max_dist)
        # "none"       -> raw distance (no accuracy)
        "convert": "exp",

        # exp conversion parameter (higher = more forgiving)
        # typical range: 0.02 – 0.2 if coords are normalized
        "alpha": 0.05,

        # threshold for "threshold" or "linear" mode
        "threshold": 0.1,

        # resampling target
        # usually same as model output length
        "target_frames": 20,

        # whether to weight joints differently
        "use_joint_weights": True,

        # joint weights (length = 75)
        # pose joints lighter, hands heavier
        "joint_weights": {
            "pose": 0.5,     # 33 joints
            "left_hand": 1.0, # 21 joints
            "right_hand": 1.0 # 21 joints
        },

        # cache GT sequences in memory for speed
        "preload_gt": True,

        # compute this metric on validation only
        "compute_on": "val",  # "train" | "val" | "both"

        # log per-gloss breakdown (expensive but informative)
        "log_per_gloss": False
    }
}


os.makedirs(CONFIG['save_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

In [6]:

def inspect_npy(file_path):
    arr = np.load(file_path, allow_pickle=True)
    
    print("Shape:", arr.shape)
    print("Dtype:", arr.dtype)
    print("Number of dimensions:", arr.ndim)
    print("Size (total elements):", arr.size)
    print("First few elements:", arr.flat[:10])

inspect_npy(r"E:\WLASL\wlasl_1000_preproc\videos\1\01610.npy")

Shape: (125, 75, 3)
Dtype: float16
Number of dimensions: 3
Size (total elements): 28125
First few elements: [-0.01079 -0.2498  -0.678    0.01654 -0.285   -0.628    0.02985 -0.2837
 -0.6284   0.0393 ]


In [11]:
import numpy as np
import os
from glob import glob
import json
from collections import Counter

FPS = 30
CONF_THRESHOLD = 0.3

POSE_IDX = slice(0, 33)
LEFT_IDX = slice(33, 54)
RIGHT_IDX = slice(54, 75)

WRIST = 0
MIDDLE_TIP = 12

def euclidean(a, b):
    return np.linalg.norm(a - b)

def normalize(v):
    return v / (np.linalg.norm(v) + 1e-6)

# ============================
# HAND FEATURES
# ============================

def finger_extension(hand):
    wrist = hand[WRIST][:2]
    tips = [4, 8, 12, 16, 20]
    dists = [euclidean(hand[t][:2], wrist) for t in tips]
    max_dist = max(dists) + 1e-6
    return np.array(dists) / max_dist

def flexion_level(ext):
    return float(np.mean(ext))

def classify_handshape(ext):
    m = np.mean(ext)
    if m > 0.75:
        return "open"
    elif m < 0.3:
        return "fist"
    elif ext[1] > 0.7 and np.mean(ext[2:]) < 0.4:
        return "index"
    return "other"

# ============================
# ORIENTATION
# ============================

def palm_orientation(hand):
    wrist = hand[WRIST][:2]
    middle = hand[MIDDLE_TIP][:2]

    vec = normalize(middle - wrist)

    if abs(vec[0]) > abs(vec[1]):
        return "palm_right" if vec[0] > 0 else "palm_left"
    else:
        return "palm_down" if vec[1] > 0 else "palm_up"

# ============================
# LOCATION
# ============================

def get_torso_center(pose):
    return (pose[11][:2] + pose[12][:2]) / 2

def location_label(hand, pose):
    ref = get_torso_center(pose)
    pos = hand[WRIST][:2]

    dy = pos[1] - ref[1]

    if dy < -0.4:
        return "head"
    elif dy < -0.15:
        return "upper"
    elif dy < 0.2:
        return "chest"
    return "torso"

def location_relative(hand, pose):
    ref = get_torso_center(pose)
    return (hand[WRIST][:2] - ref).tolist()

# ============================
# MOVEMENT
# ============================

def compute_velocity(traj):
    return np.diff(traj, axis=0)

def compute_speed(traj):
    vel = compute_velocity(traj)
    return float(np.mean(np.linalg.norm(vel, axis=1)))

def amplitude(traj):
    return float(np.max(np.linalg.norm(traj - traj[0], axis=1)))

def movement_direction(traj):
    delta = traj[-1] - traj[0]
    if abs(delta[0]) > abs(delta[1]):
        return "horizontal"
    else:
        return "vertical"

def movement_type(traj):
    vel = compute_velocity(traj)
    changes = np.sum(np.linalg.norm(np.diff(vel, axis=0), axis=1))

    if changes < 0.05:
        return "linear"
    elif changes < 0.2:
        return "arc"
    return "complex"

def repetition_count(traj):
    vel = compute_velocity(traj)
    signs = np.sign(vel[:, 0])
    return int(np.sum(np.abs(np.diff(signs)) > 0) + 1)

# ============================
# TWO HAND
# ============================

def detect_contact(l, r):
    return bool(np.any(np.linalg.norm(l - r, axis=1) < 0.05))

def symmetry(l, r):
    dl = l[-1] - l[0]
    dr = r[-1] - r[0]
    return "symmetric" if np.dot(dl, dr) > 0 else "asymmetric"

# ============================
# FLAGS
# ============================

def detect_flags(traj):
    amp = amplitude(traj)
    reps = repetition_count(traj)

    return {
        "plural": reps > 2,
        "negation": False,  # cannot infer reliably
        "intense": amp > 0.25
    }

# ============================

def extract_lsmu(seq):
    T = seq.shape[0]

    left_traj = []
    right_traj = []
    handshapes = []
    orientations = []
    locations = []
    confidences = []

    for t in range(T):
        frame = seq[t]

        pose = frame[POSE_IDX]
        left = frame[LEFT_IDX]
        right = frame[RIGHT_IDX]

        conf = np.mean(left[:, 2])
        if conf < CONF_THRESHOLD:
            continue

        left_traj.append(left[WRIST][:2])
        right_traj.append(right[WRIST][:2])

        ext = finger_extension(left)

        handshapes.append(classify_handshape(ext))
        orientations.append(palm_orientation(left))
        locations.append(location_label(left, pose))
        confidences.append(conf)

    if len(left_traj) < 5:
        return None

    left_traj = np.array(left_traj)
    right_traj = np.array(right_traj)

    # =======================
    # BUILD FINAL STRUCTURE
    
    movement = movement_direction(left_traj)

    lsmu = {
        "handshape": max(set(handshapes), key=handshapes.count),
        "orientation": max(set(orientations), key=orientations.count),
        "location": max(set(locations), key=locations.count),

        "movement": movement,

        "location_relative": np.mean(left_traj, axis=0).tolist(),

        "amplitude": amplitude(left_traj),
        "speed": compute_speed(left_traj),
        "repetition_count": repetition_count(left_traj),
        "phase_count": 1,

        "symmetry": symmetry(left_traj, right_traj),
        "contact": detect_contact(left_traj, right_traj),

        "confidence": float(np.mean(confidences)),

        "flags": detect_flags(left_traj)
    }

    return lsmu


# =======================================

LABEL_MAP_PATH = r"E:\WLASL\wlasl_1000_preproc\label_map_final.json"

def load_label_map(path):
    with open(path, "r") as f:
        data = json.load(f)

    # Case 1: {"1": "book"}
    if all(isinstance(k, str) and isinstance(v, str) for k, v in data.items()):
        return data

    # Case 2: {"book": 1} → reverse it
    if all(isinstance(v, int) for v in data.values()):
        return {str(v): k for k, v in data.items()}

    # Case 3: nested formats (WLASL sometimes does weird stuff)
    if isinstance(data, list):
        # Example: [{"gloss": "book", "id": 1}, ...]
        mapping = {}
        for item in data:
            if "gloss" in item and "id" in item:
                mapping[str(item["id"])] = item["gloss"]
        return mapping

    raise ValueError("Unsupported label_map format")

def aggregate_lsmu(lsmu_list):
    """
    Aggregate multiple samples into one gloss-level LSMU
    """
    if len(lsmu_list) == 0:
        return None

    agg = {}

    # categorical → majority vote
    cat_keys = ["handshape", "orientation", "location", "movement", "symmetry"]

    for key in cat_keys:
        values = [x[key] for x in lsmu_list if key in x]
        if values:
            agg[key] = Counter(values).most_common(1)[0][0]

    # numeric → mean
    num_keys = ["amplitude", "speed", "confidence"]

    for key in num_keys:
        values = [x[key] for x in lsmu_list if key in x]
        if values:
            agg[key] = float(np.mean(values))

    # repetition → median (better than mean)
    reps = [x["repetition_count"] for x in lsmu_list if "repetition_count" in x]
    if reps:
        agg["repetition_count"] = int(np.median(reps))

    # contact → OR
    agg["contact"] = any(x.get("contact", False) for x in lsmu_list)

    # flags → OR logic
    flags = {
        "plural": any(x["flags"]["plural"] for x in lsmu_list),
        "negation": any(x["flags"]["negation"] for x in lsmu_list),
        "intense": any(x["flags"]["intense"] for x in lsmu_list),
    }
    agg["flags"] = flags

    return agg


# ============================
# MAIN PROCESS
# ============================

def process_all_classes(root):
    class_dirs = sorted(os.listdir(root))

    all_lsmu = []

    for cls in class_dirs:
        class_path = os.path.join(root, cls)

        if not os.path.isdir(class_path):
            continue

        npy_files = glob(os.path.join(class_path, "*.npy"))

        lsmu_list = []

        for f in npy_files:
            try:
                seq = np.load(f)
                lsmu = extract_lsmu(seq)

                if lsmu:
                    lsmu_list.append(lsmu)

            except Exception as e:
                print(f"Error in {f}: {e}")

        if len(lsmu_list) == 0:
            print(f"[SKIP] {cls} → no valid samples")
            continue

        # aggregate
        agg_lsmu = aggregate_lsmu(lsmu_list)
        label_map = load_label_map(LABEL_MAP_PATH)
        # add word label
        word = label_map.get(cls, f"unknown_{cls}")

        output = {
            "word": word,
            "class_id": cls,
            "num_samples": len(lsmu_list),
            "lsmu": agg_lsmu
        }
        
        all_lsmu.append(output)

        # save inside the same folder
        output_path = os.path.join(class_path, "lsmu.json")

        with open(output_path, "w") as f:
            json.dump(output, f, indent=2)

        print(f"[DONE] {cls} ({word}) → {len(lsmu_list)} samples")
        
    root_parent = os.path.dirname(root)  
    final_path = os.path.join(root_parent, "final_lsmu.json")

    with open(final_path, "w") as f:
        json.dump(all_lsmu, f, indent=2)

    print(f"\n[GLOBAL SAVED] → {final_path}")



if __name__ == "__main__":
    ROOT = r"E:\WLASL\wlasl_1000_preproc\videos"
    process_all_classes(ROOT)

[DONE] 1 (a) → 1 samples
[DONE] 10 (accomplish) → 1 samples
[DONE] 100 (article) → 3 samples
[DONE] 1000 (lazy) → 4 samples
[DONE] 101 (artist) → 4 samples
[DONE] 102 (asia) → 3 samples
[DONE] 103 (ask) → 5 samples
[DONE] 104 (asl) → 3 samples
[DONE] 105 (assist) → 4 samples
[DONE] 106 (assistant) → 2 samples
[DONE] 107 (assume) → 3 samples
[DONE] 108 (attend) → 4 samples
[DONE] 109 (attention) → 4 samples
[DONE] 11 (accountant) → 2 samples
[DONE] 110 (attitude) → 5 samples
[DONE] 111 (attorney) → 3 samples
[DONE] 112 (attract) → 2 samples
[DONE] 113 (auction) → 4 samples
[DONE] 114 (audience) → 3 samples
[DONE] 115 (audiologist) → 3 samples
[DONE] 116 (audiology) → 2 samples
[DONE] 117 (august) → 4 samples
[DONE] 118 (aunt) → 5 samples
[DONE] 119 (australia) → 4 samples
[DONE] 12 (across) → 2 samples
[DONE] 120 (austria) → 2 samples
[DONE] 121 (author) → 4 samples
[DONE] 122 (authority) → 2 samples
[DONE] 123 (autumn) → 3 samples
[DONE] 124 (available) → 2 samples
[DONE] 125 (average)